In [ ]:
# Install the jaxhps library and its dependencies (skip if already installed)
%pip install -q jaxhps jax scipy matplotlib numpy

# HPS Solvers for 2D Linear Elasticity
**18.336 Fast Methods for PDEs — Georgios Tsilimidos**

This notebook demonstrates the application of the **Hierarchical Poincaré–Steklov (HPS)** fast direct solver to **2D plane-strain linear elasticity**, governed by the Navier–Cauchy system:

$$\begin{pmatrix}(\lambda+2\mu)\partial_{xx}+\mu\partial_{yy} & (\lambda+\mu)\partial_{xy}\\(\lambda+\mu)\partial_{xy} & \mu\partial_{xx}+(\lambda+2\mu)\partial_{yy}\end{pmatrix}\begin{pmatrix}u_x\\u_y\end{pmatrix}=-\mathbf{f}, \qquad \mathbf{u}\big|_{\partial\Omega}=\mathbf{g}$$

Two benchmarks are studied:
1. **Smooth manufactured solution** — spectral-exponential convergence in polynomial degree $p$.
2. **Mode I crack-tip singularity** — algebraic $\mathcal{O}(p^{-1/2})$ convergence due to the Williams $\sqrt{r}$ singularity.

All solver logic lives in `examples/linear_elasticity_2D.py` and `examples/linear_elasticity_crack_tip_2D.py`; this notebook imports and calls those functions.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "examples"))

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

jax.config.update("jax_default_device", jax.devices("cpu")[0])

## Implementation Overview

The implementation is split into two self-contained scripts:

### `examples/linear_elasticity_2D.py`
Solves the Navier–Cauchy equations on $[-1,1]^2$ with a **smooth manufactured solution**.

Key functions:
- `ux_exact`, `uy_exact` — exact solution ($\sin\pi x\sin\pi y$, $\cos\pi x\cos\pi y$)
- `fx_body`, `fy_body` — corresponding body forces derived from equilibrium
- **`gauss_seidel_elasticity(domain, max_iter, tol)`** — the main ADI solver (see below)
- `run_convergence(l_vals, p_vals, ...)` — hp-sweep returning error and timing arrays

### `examples/linear_elasticity_crack_tip_2D.py`
Same structure for the **Mode I crack-tip** problem on $[0,1]^2$.

Key functions:
- `williams_ux`, `williams_uy` — Williams asymptotic expansion ($K_I=\mu=1$, $\kappa=2$)
- **`gauss_seidel_crack_tip(domain, max_iter, tol)`** — same ADI solver, zero body force
- `run_convergence(l_vals, p_vals, ...)` — hp-sweep

### ADI splitting (shared strategy)

Because `jaxhps` is a scalar solver, the coupled Navier–Cauchy system is decoupled via an **alternating-direction iteration**. At step $k$:

$$[(\lambda+2\mu)\partial_{xx}+\mu\partial_{yy}]\,u_x^{(k)} = -f_x - (\lambda+\mu)\,\partial_{xy}\,u_y^{(k-1)}$$
$$[\mu\partial_{xx}+(\lambda+2\mu)\partial_{yy}]\,u_y^{(k)} = -f_y - (\lambda+\mu)\,\partial_{xy}\,u_x^{(k)}$$

The differential operators are **fixed** across iterations, so `build_solver` (the expensive HPS build stage) is called **once per component**. Each ADI step then calls only the cheap `solve` (HPS downward pass) plus a spectral cross-derivative product.

## Case 1: Smooth Manufactured Solution

**Domain:** $[-1,1]^2$, $\lambda=\mu=1$.  
**Exact solution:** $u_x = \sin(\pi x)\sin(\pi y)$, $u_y = \cos(\pi x)\cos(\pi y)$ — analytic, so **spectral-exponential** convergence in $p$ is expected and observed.

In [ ]:
from linear_elasticity_2D import (
    gauss_seidel_elasticity,
    run_convergence,
    ux_exact, uy_exact,
    XMIN, XMAX, YMIN, YMAX,
)
from jaxhps import DiscretizationNode2D, Domain

# --- Single demonstration solve at L=3, p=8 ---
root   = DiscretizationNode2D(xmin=XMIN, xmax=XMAX, ymin=YMIN, ymax=YMAX)
domain = Domain(p=8, q=6, root=root, L=3)

u_x, u_y, residuals, t_build, t_solve = gauss_seidel_elasticity(domain)

err_ux = float(jnp.max(jnp.abs(u_x - ux_exact(domain.interior_points)))
               / jnp.max(jnp.abs(ux_exact(domain.interior_points))))
err_uy = float(jnp.max(jnp.abs(u_y - uy_exact(domain.interior_points)))
               / jnp.max(jnp.abs(uy_exact(domain.interior_points))))

print(f"L=3, p=8  |  rel L∞ error:  u_x = {err_ux:.2e},  u_y = {err_uy:.2e}")
print(f"ADI iters: {len(residuals)}  |  build: {t_build:.2f}s  |  solve: {t_solve:.2f}s")

# --- hp-convergence sweep (adjust p_vals / l_vals to trade accuracy for speed) ---
p_vals = [4, 8, 12, 16]
l_vals = [2, 3]

print("\nhp-convergence sweep:")
results = run_convergence(l_vals, p_vals, max_iter=30, tol=1e-14)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# --- Convergence in p for each level L ---
ax = axes[0]
linestyles = ["-o", "-s"]
for i, L in enumerate(l_vals):
    errors = results["errors_ux"][i]          # shape (len(p_vals),)
    ax.semilogy(p_vals, errors, linestyles[i], label=f"$L={L}$")
ax.set_xlabel("polynomial degree $p$")
ax.set_ylabel(r"rel. $L^\infty$ error  $u_x$")
ax.set_title("Smooth: spectral convergence in $p$")
ax.legend()
ax.grid(True, which="both", alpha=0.3)

# --- Solution contour at last (finest) domain ---
ax2 = axes[1]
dom_last = results["last_domain"]
pts = dom_last.interior_points
x_pts = np.array(pts[:, 0])
y_pts = np.array(pts[:, 1])
u_plot = np.array(results["last_ux"])

sc = ax2.tricontourf(x_pts, y_pts, u_plot, levels=20, cmap="RdBu_r")
plt.colorbar(sc, ax=ax2, label=r"$u_x$")
ax2.set_title(f"Computed $u_x$ (finest solve)")
ax2.set_aspect("equal")

fig.tight_layout()
plt.show()

## Case 2: Mode I Crack-Tip Singularity

**Domain:** $[0,1]^2$, $\lambda=\mu=1$, stress-intensity factor $K_I=1$.  
**Exact solution:** Williams asymptotic expansion for a Mode I crack lying along the negative $x$-axis,

$$u_r = \frac{K_I}{2\mu}\sqrt{\frac{r}{2\pi}}\Bigl[\kappa-\cos\theta\Bigr]\cos\tfrac{\theta}{2}, \qquad u_\theta = \frac{K_I}{2\mu}\sqrt{\frac{r}{2\pi}}\Bigl[\kappa+\cos\theta\Bigr]\sin\tfrac{\theta}{2}$$

with $\kappa = 3-4\nu$ (plane-strain) and $\nu=1/4$.  The $\sqrt{r}$ singularity at the origin limits convergence to **algebraic** $\mathcal{O}(p^{-1/2})$.

In [ ]:
from linear_elasticity_crack_tip_2D import (
    gauss_seidel_crack_tip,
    run_convergence as run_convergence_crack,
    williams_ux, williams_uy,
    XMIN as XMIN_C, XMAX as XMAX_C, YMIN as YMIN_C, YMAX as YMAX_C,
)

# --- Single demonstration solve ---
root_c   = DiscretizationNode2D(xmin=XMIN_C, xmax=XMAX_C, ymin=YMIN_C, ymax=YMAX_C)
domain_c = Domain(p=8, q=6, root=root_c, L=3)

ux_c, uy_c, residuals_c, t_build_c, t_solve_c = gauss_seidel_crack_tip(domain_c)

err_ux_c = float(jnp.max(jnp.abs(ux_c - williams_ux(domain_c.interior_points)))
                 / jnp.max(jnp.abs(williams_ux(domain_c.interior_points))))
err_uy_c = float(jnp.max(jnp.abs(uy_c - williams_uy(domain_c.interior_points)))
                 / jnp.max(jnp.abs(williams_uy(domain_c.interior_points))))

print(f"L=3, p=8  |  rel L∞ error:  u_x = {err_ux_c:.2e},  u_y = {err_uy_c:.2e}")
print(f"ADI iters: {len(residuals_c)}  |  build: {t_build_c:.2f}s  |  solve: {t_solve_c:.2f}s")

# --- hp-convergence sweep ---
p_vals_c = [4, 8, 12, 16]
l_vals_c = [2, 3]

print("\nhp-convergence sweep:")
results_c = run_convergence_crack(l_vals_c, p_vals_c, max_iter=30, tol=1e-14)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# --- Log-log convergence in p ---
ax = axes[0]
for i, L in enumerate(l_vals_c):
    errors = results_c["errors_ux"][i]
    ax.loglog(p_vals_c, errors, linestyles[i], label=f"$L={L}$")

# Reference slope O(p^{-1/2})
p_arr = np.array(p_vals_c, dtype=float)
ref = 0.3 * p_arr**(-0.5)
ax.loglog(p_arr, ref, "k--", label=r"$\mathcal{O}(p^{-1/2})$")

ax.set_xlabel("polynomial degree $p$")
ax.set_ylabel(r"rel. $L^\infty$ error  $u_x$")
ax.set_title("Crack-tip: algebraic convergence in $p$")
ax.legend()
ax.grid(True, which="both", alpha=0.3)

# --- Solution contour ---
ax2 = axes[1]
dom_c_last = results_c["last_domain"]
pts_c = dom_c_last.interior_points
sc2 = ax2.tricontourf(
    np.array(pts_c[:, 0]),
    np.array(pts_c[:, 1]),
    np.array(results_c["last_ux"]),
    levels=20, cmap="RdBu_r",
)
plt.colorbar(sc2, ax=ax2, label=r"$u_x$")
ax2.set_title("Computed $u_x$ — crack-tip (finest solve)")
ax2.set_aspect("equal")

fig.tight_layout()
plt.show()